# Track C — eval harness, fusion & submission

**Upload `trackc.py` and this notebook into the Jupyter folder that contains `data/`, then Run All.**

This notebook is the team's measurement instrument. It:
1. loads the data manifest (checks 377 submission rows / 350 labeled pairs),
2. builds the 300/50 dataset1 split + a synthetic Level-2 (deformation) proxy,
3. measures **local MRR** with a placeholder baseline branch (replaced by B1/B2/B3/B4),
4. fuses branch rankings with **RRF** and writes a validated `submission.csv`.

Paste the printed outputs back to the chat so we can verify against Kaggle.

In [ ]:
import importlib, trackc
importlib.reload(trackc)
from trackc import *
import numpy as np, pandas as pd

DATA_ROOT = "data"          # edit if your data folder is elsewhere
trackc.DATA_ROOT = DATA_ROOT
trackc._selftest()           # metric/RRF sanity (no data needed)

## 1. Manifest — verify the data contract (expect 377 rows, 350 pairs)

In [ ]:
manifest = load_manifest(DATA_ROOT)
_ = manifest_summary(manifest)

## 2. Local 300/50 split + ground truth
The held-out 50 dataset1 pairs form a self-contained retrieval problem: rank each held-out query against the 50 held-out targets. This is our offline Level-1 MRR proxy.

In [ ]:
train_df, hold_df, gt = make_local_split(manifest['train_pairs'], n_holdout=50, seed=0)
print('train pairs:', len(train_df), '| held-out:', len(hold_df))
print('held-out queries are ranked against', len(hold_df), 'targets')
hold_df.head(3)

## 3. Baseline branch → Level-1 local MRR
`embed_baseline` is a cheap placeholder (gradient-magnitude of a downsampled volume). The number it prints calibrates the harness; B1/B2/B3/B4 will replace it. Random chance ≈ 1/50 = 0.02.

In [ ]:
def embed_pool(df, id_col, img_col):
    emb = {}
    for _, r in df.iterrows():
        emb[r[id_col]] = embed_baseline(r[img_col], data_root=DATA_ROOT)
    return emb

q_emb = embed_pool(hold_df, 'query_id', 'query_image')
g_emb = embed_pool(hold_df, 'target_id', 'target_image')
rank_l1 = rank_by_embeddings(q_emb, g_emb)
print('Level-1 local MRR (baseline) =', round(mrr(rank_l1, gt), 4))

## 4. Synthetic Level-2 proxy — deform the held-out targets, re-measure
Independent random rigid + nonlinear deformation on each held-out target. This is the only way to tune deformation-invariance offline (d2/d3 have no labels). Loads + warps 50 volumes — give it a minute.

In [ ]:
from scipy.ndimage import zoom
rng = np.random.default_rng(0)
g_emb_l2 = {}
for _, r in hold_df.iterrows():
    vol = load_volume(r['target_image'], resample_1mm=False, zscore=True, data_root=DATA_ROOT)
    vol = resize_to(vol, (48, 48, 48))
    vol = deform_volume(vol, rng=rng)
    gx, gy, gz = np.gradient(vol)
    v = np.sqrt(gx**2 + gy**2 + gz**2).ravel().astype(np.float32)
    g_emb_l2[r['target_id']] = v / (np.linalg.norm(v) + 1e-8)
rank_l2 = rank_by_embeddings(q_emb, g_emb_l2)
print('Level-2 proxy local MRR (baseline) =', round(mrr(rank_l2, gt), 4))

## 5. Full pipeline → validated submission.csv
Rank every (dataset, split) pool with the baseline branch, fuse (single branch here = identity), validate, write. This is the round-trip that proves the submission file is byte-correct before we ever spend a Kaggle submission.

In [ ]:
pool_rankings = {}
for (ds, split), pool in [(k, v) for k, v in manifest.items() if k != 'train_pairs']:
    qe = embed_pool(pool.queries, 'query_id', 'query_image')
    ge = embed_pool(pool.gallery, 'target_id', 'target_image')
    branch = rank_by_embeddings(qe, ge)
    # RRF over the list of branches for this pool (just one for now):
    pool_rankings[(ds, split)] = rrf([branch])
    print(ds, split, 'ranked', len(branch), 'queries')

sub = write_submission(pool_rankings, manifest, path='submission.csv')
validate_submission(sub, manifest)
sub.head(3)

## Paste back to chat
- the **manifest summary** table (cell 1),
- the two **local MRR** numbers (cells 3 & 4),
- the **`submission OK: ...`** line (cell 5).

Then we plug in real branches: B2 (foundation embeddings, my track) and B1/B3/B4 from teammates — each just emits `[query_id, target_id, score]`, and RRF + this notebook do the rest.